In [10]:
import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import joblib

# ==========================
# PARAMETERS
# ==========================

FRAME_SIZE = 0.025
HOP_SIZE = 0.01
N_FFT = 1024
LOW_FREQ = 200
HIGH_FREQ = 500
N_MFCC = 13


# ==========================
# FEATURE EXTRACTION
# ==========================

def extract_frame_features(y, sr):

    frame_length = int(FRAME_SIZE * sr)
    hop_length = int(HOP_SIZE * sr)

    S = np.abs(librosa.stft(
        y,
        n_fft=N_FFT,
        hop_length=hop_length,
        win_length=frame_length
    ))

    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
    band_mask = (freqs >= LOW_FREQ) & (freqs <= HIGH_FREQ)

    total_energy = np.sum(S, axis=0) + 1e-8
    band_energy = np.sum(S[band_mask, :], axis=0)

    band_ratio = band_energy / total_energy
    centroid = librosa.feature.spectral_centroid(S=S, sr=sr)[0]
    flatness = librosa.feature.spectral_flatness(S=S)[0]
    rolloff = librosa.feature.spectral_rolloff(S=S, sr=sr)[0]

    zcr = librosa.feature.zero_crossing_rate(
        y,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    psd = S / total_energy
    entropy = -np.sum(psd * np.log(psd + 1e-8), axis=0)

    rms = librosa.feature.rms(
        y=y,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    mfccs = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=hop_length,
        win_length=frame_length
    )

    delta_mfccs = librosa.feature.delta(mfccs)

    chroma = librosa.feature.chroma_stft(
        S=S,
        sr=sr,
        n_fft=N_FFT,
        hop_length=hop_length
    )

    contrast = librosa.feature.spectral_contrast(
        S=S,
        sr=sr,
        n_fft=N_FFT,
        hop_length=hop_length,
        fmin=10.0,
        n_bands=4
    )

    tonnetz = librosa.feature.tonnetz(chroma=chroma)

    n_frames = S.shape[1]

    def _trim(arr):
        if arr.ndim == 1:
            return arr[:n_frames]
        return arr[:, :n_frames]

    band_ratio  = _trim(band_ratio)
    centroid    = _trim(centroid)
    flatness    = _trim(flatness)
    rolloff     = _trim(rolloff)
    zcr         = _trim(zcr)
    entropy     = _trim(entropy)
    rms         = _trim(rms)
    mfccs       = _trim(mfccs)
    delta_mfccs = _trim(delta_mfccs)
    chroma      = _trim(chroma)
    contrast    = _trim(contrast)
    tonnetz     = _trim(tonnetz)

    features = np.vstack([
        band_ratio,
        centroid,
        flatness,
        rolloff,
        zcr,
        entropy,
        rms,
        mfccs,
        delta_mfccs,
        chroma,
        contrast,
        tonnetz
    ]).T

    return features, hop_length


def _build_column_names():
    cols = [
        "band_ratio", "centroid", "flatness", "rolloff",
        "zcr", "entropy", "rms"
    ]
    cols += [f"mfcc_{i+1}" for i in range(N_MFCC)]
    cols += [f"delta_mfcc_{i+1}" for i in range(N_MFCC)]
    cols += [f"chroma_{i+1}" for i in range(12)]
    cols += [f"contrast_{i+1}" for i in range(5)]
    cols += [f"tonnetz_{i+1}" for i in range(6)]
    return cols

FEATURE_COLUMNS = _build_column_names()


# ==========================
# LABEL ASSIGNMENT
# ==========================

def create_frame_labels(label_file, total_frames, sr, hop_length):

    df = pd.read_csv(label_file,
                     sep="\t",
                     header=None,
                     names=["start", "end", "label"])

    labels = np.zeros(total_frames)

    for _, row in df.iterrows():
        if str(row["label"]).strip() == "Rhonchi":
            start_frame = int((row["start"] * sr) / hop_length)
            end_frame = int((row["end"] * sr) / hop_length)

            start_frame = max(0, start_frame)
            end_frame = min(total_frames, end_frame)

            labels[start_frame:end_frame] = 1

    return labels


# ==========================
# DATASET BUILDER
# ==========================

def build_dataset(audio_folder):

    X_all = []
    y_all = []

    audio_files = sorted(os.listdir(audio_folder))

    for file in tqdm(audio_files):

        if not file.endswith(".wav"):
            continue

        audio_path = os.path.join(audio_folder, file)

        base_name = file.replace(".wav", "")
        label_file = base_name + "_label_audacity.txt"
        label_path = os.path.join(audio_folder, label_file)

        if not os.path.exists(label_path):
            print(f"⚠ Missing label for {file}")
            continue

        y, sr = librosa.load(audio_path, sr=None)

        features, hop_length = extract_frame_features(y, sr)

        labels = create_frame_labels(
            label_path,
            features.shape[0],
            sr,
            hop_length
        )

        X_all.append(features)
        y_all.append(labels)

    X = np.vstack(X_all)
    y = np.hstack(y_all)

    return X, y


def post_process_predictions(predictions,
                             hop_size_sec,
                             min_duration_sec=0.15,
                             max_gap_frames=8):

    cleaned = predictions.copy()

    i = 0
    while i < len(cleaned):

        if cleaned[i] == 0:
            gap_start = i

            while i < len(cleaned) and cleaned[i] == 0:
                i += 1

            gap_end = i
            gap_length = gap_end - gap_start

            if (gap_length <= max_gap_frames and
                gap_start > 0 and
                gap_end < len(cleaned) and
                cleaned[gap_start - 1] == 1 and
                cleaned[gap_end] == 1):

                cleaned[gap_start:gap_end] = 1

        else:
            i += 1

    min_frames = int(min_duration_sec / hop_size_sec)
    start = None

    for i in range(len(cleaned)):

        if cleaned[i] == 1 and start is None:
            start = i

        if (cleaned[i] == 0 or i == len(cleaned)-1) and start is not None:

            end = i if cleaned[i] == 0 else i + 1

            if end - start < min_frames:
                cleaned[start:end] = 0

            start = None

    return cleaned


# ==========================
# MAIN
# ==========================

def main():

    audio_folder = r"E:\Research Project (Prof. Preeti Rao)\Top files by Rhonchi count"

    print("Building dataset...")
    X, y = build_dataset(audio_folder)

    full_df = pd.DataFrame(X, columns=FEATURE_COLUMNS)
    full_df["label"] = y
    full_df.to_csv("full_dataset.csv", index=False)

    print("Full dataset saved.")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    clf = RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    )

    print("Training...")
    clf.fit(X_train, y_train)

    print("Predicting...")
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]

    y_pred_clean = post_process_predictions(
        y_pred,
        hop_size_sec=HOP_SIZE,
        min_duration_sec=0.20,
        max_gap_frames=20
    )

    train_df = pd.DataFrame(X_train, columns=FEATURE_COLUMNS)
    train_df["label"] = y_train
    train_df.to_csv("train_data.csv", index=False)

    test_df = pd.DataFrame(X_test, columns=FEATURE_COLUMNS)
    test_df["label"] = y_test
    test_df["predicted_label"] = y_pred
    test_df["predicted_probability"] = y_prob

    test_df.to_csv("test_data_with_predictions.csv", index=False)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    joblib.dump(clf, "rhonchi_model.pkl")
    joblib.dump(scaler, "scaler.pkl")

    print("\nEverything saved successfully.")


if __name__ == "__main__":
    main()

Building dataset...


  0%|          | 0/218 [00:00<?, ?it/s]

100%|██████████| 218/218 [00:08<00:00, 24.56it/s]


Full dataset saved.
Training...
Predicting...

Confusion Matrix:
[[13196  3275]
 [ 4184 12067]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.76      0.80      0.78     16471
         1.0       0.79      0.74      0.76     16251

    accuracy                           0.77     32722
   macro avg       0.77      0.77      0.77     32722
weighted avg       0.77      0.77      0.77     32722


Everything saved successfully.


In [11]:
import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

# ==========================
# PARAMETERS
# ==========================

FRAME_SIZE = 0.025
HOP_SIZE = 0.01
N_FFT = 1024
LOW_FREQ = 200
HIGH_FREQ = 500
MIN_DURATION = 0.15
N_MFCC = 13


# ==========================
# FEATURE EXTRACTION
# ==========================

def extract_features_with_time(y, sr):

    frame_length = int(FRAME_SIZE * sr)
    hop_length = int(HOP_SIZE * sr)

    S = np.abs(librosa.stft(
        y,
        n_fft=N_FFT,
        hop_length=hop_length,
        win_length=frame_length
    ))

    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
    band_mask = (freqs >= LOW_FREQ) & (freqs <= HIGH_FREQ)

    total_energy = np.sum(S, axis=0) + 1e-8
    band_energy = np.sum(S[band_mask, :], axis=0)

    band_ratio = band_energy / total_energy
    centroid = librosa.feature.spectral_centroid(S=S, sr=sr)[0]
    flatness = librosa.feature.spectral_flatness(S=S)[0]
    rolloff = librosa.feature.spectral_rolloff(S=S, sr=sr)[0]

    zcr = librosa.feature.zero_crossing_rate(
        y,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    psd = S / total_energy
    entropy = -np.sum(psd * np.log(psd + 1e-8), axis=0)

    rms = librosa.feature.rms(
        y=y,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    mfccs = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=hop_length,
        win_length=frame_length
    )

    delta_mfccs = librosa.feature.delta(mfccs)

    chroma = librosa.feature.chroma_stft(
        S=S,
        sr=sr,
        n_fft=N_FFT,
        hop_length=hop_length
    )

    contrast = librosa.feature.spectral_contrast(
        S=S,
        sr=sr,
        n_fft=N_FFT,
        hop_length=hop_length,
        fmin=10.0,
        n_bands=4
    )

    tonnetz = librosa.feature.tonnetz(chroma=chroma)

    n_frames = S.shape[1]

    def _trim(arr):
        if arr.ndim == 1:
            return arr[:n_frames]
        return arr[:, :n_frames]

    band_ratio   = _trim(band_ratio)
    centroid     = _trim(centroid)
    flatness     = _trim(flatness)
    rolloff      = _trim(rolloff)
    zcr          = _trim(zcr)
    entropy      = _trim(entropy)
    rms          = _trim(rms)
    mfccs        = _trim(mfccs)
    delta_mfccs  = _trim(delta_mfccs)
    chroma       = _trim(chroma)
    contrast     = _trim(contrast)
    tonnetz      = _trim(tonnetz)

    start_times = np.arange(n_frames) * HOP_SIZE
    end_times = start_times + FRAME_SIZE

    features = np.vstack([
        band_ratio,
        centroid,
        flatness,
        rolloff,
        zcr,
        entropy,
        rms,
        mfccs,
        delta_mfccs,
        chroma,
        contrast,
        tonnetz
    ]).T

    return features, start_times, end_times, hop_length


def _build_column_names():
    cols = [
        "band_ratio", "centroid", "flatness", "rolloff",
        "zcr", "entropy", "rms"
    ]
    cols += [f"mfcc_{i+1}" for i in range(N_MFCC)]
    cols += [f"delta_mfcc_{i+1}" for i in range(N_MFCC)]
    cols += [f"chroma_{i+1}" for i in range(12)]
    cols += [f"contrast_{i+1}" for i in range(5)]
    cols += [f"tonnetz_{i+1}" for i in range(6)]
    return cols

FEATURE_COLUMNS = _build_column_names()


# ==========================
# LABEL ASSIGNMENT
# ==========================

def create_frame_labels(label_file, total_frames, sr, hop_length):

    df = pd.read_csv(label_file,
                     sep="\t",
                     header=None,
                     names=["start", "end", "label"])

    labels = np.zeros(total_frames)

    for _, row in df.iterrows():
        if str(row["label"]).strip() == "Rhonchi":
            start_frame = int((row["start"] * sr) / hop_length)
            end_frame = int((row["end"] * sr) / hop_length)

            start_frame = max(0, start_frame)
            end_frame = min(total_frames, end_frame)

            labels[start_frame:end_frame] = 1

    return labels


# ==========================
# TEMPORAL CONSTRAINT
# ==========================

def post_process_predictions(predictions,
                             hop_size_sec,
                             min_duration_sec=0.20,
                             max_gap_frames=20):

    cleaned = predictions.copy()

    i = 0
    while i < len(cleaned):

        if cleaned[i] == 0:
            gap_start = i

            while i < len(cleaned) and cleaned[i] == 0:
                i += 1

            gap_end = i
            gap_length = gap_end - gap_start

            if (gap_length <= max_gap_frames and
                gap_start > 0 and
                gap_end < len(cleaned) and
                cleaned[gap_start - 1] == 1 and
                cleaned[gap_end] == 1):

                cleaned[gap_start:gap_end] = 1

        else:
            i += 1

    min_frames = int(min_duration_sec / hop_size_sec)
    start = None

    for i in range(len(cleaned)):

        if cleaned[i] == 1 and start is None:
            start = i

        if (cleaned[i] == 0 or i == len(cleaned)-1) and start is not None:

            end = i if cleaned[i] == 0 else i + 1

            if end - start < min_frames:
                cleaned[start:end] = 0

            start = None

    return cleaned


# ==========================
# MAIN PIPELINE
# ==========================

def main():

    audio_folder = r"E:\Research Project (Prof. Preeti Rao)\Top files by Rhonchi count"
    output_folder = "output"

    os.makedirs(output_folder + "/train", exist_ok=True)
    os.makedirs(output_folder + "/test", exist_ok=True)

    audio_files = sorted([f for f in os.listdir(audio_folder)
                          if f.endswith(".wav")])

    train_files, test_files = train_test_split(
        audio_files,
        test_size=0.2,
        random_state=42
    )

    X_train_all = []
    y_train_all = []

    for file in tqdm(train_files):

        base = file.replace(".wav", "")
        label_file = base + "_label_audacity.txt"

        y, sr = librosa.load(os.path.join(audio_folder, file), sr=None)

        features, start_t, end_t, hop_length = extract_features_with_time(y, sr)
        labels = create_frame_labels(
            os.path.join(audio_folder, label_file),
            features.shape[0],
            sr,
            hop_length
        )

        X_train_all.append(features)
        y_train_all.append(labels)

        df = pd.DataFrame(features, columns=FEATURE_COLUMNS)
        df["start_time"] = start_t
        df["end_time"] = end_t
        df["true_label"] = labels

        df.to_csv(f"{output_folder}/train/{base}_train.csv", index=False)

    X_train = np.vstack(X_train_all)
    y_train = np.hstack(y_train_all)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)

    clf = RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    clf.fit(X_train, y_train)

    for file in tqdm(test_files):

        base = file.replace(".wav", "")
        label_file = base + "_label_audacity.txt"

        y, sr = librosa.load(os.path.join(audio_folder, file), sr=None)

        features, start_t, end_t, hop_length = extract_features_with_time(y, sr)
        labels = create_frame_labels(
            os.path.join(audio_folder, label_file),
            features.shape[0],
            sr,
            hop_length
        )

        X_test = scaler.transform(features)

        y_pred = clf.predict(X_test)
        y_pred_clean = post_process_predictions(
            y_pred,
            hop_size_sec=HOP_SIZE,
            min_duration_sec=0.15,
            max_gap_frames=12
        )

        df = pd.DataFrame(features, columns=FEATURE_COLUMNS)
        df["start_time"] = start_t
        df["end_time"] = end_t
        df["true_label"] = labels
        df["predicted_label_raw"] = y_pred
        df["predicted_label_0.15s"] = y_pred_clean

        df.to_csv(f"{output_folder}/test/{base}_test.csv", index=False)

    joblib.dump(clf, "rhonchi_model.pkl")
    joblib.dump(scaler, "scaler.pkl")

    print("\n✅ File-wise CSV generation complete.")


if __name__ == "__main__":
    main()

  0%|          | 0/87 [00:00<?, ?it/s]

100%|██████████| 22/22 [00:07<00:00,  2.81it/s]



✅ File-wise CSV generation complete.


In [12]:
import os
import numpy as np
import pandas as pd


# ==========================
# PARAMETERS
# ==========================

HOP_SIZE = 0.01


# ==========================
# COUNT FROM LABEL FILE
# ==========================

def count_rhonchi_from_label(label_file):

    df = pd.read_csv(label_file,
                     sep="\t",
                     header=None,
                     names=["start", "end", "label"])

    count = df[df["label"].str.strip() == "Rhonchi"].shape[0]

    return count


# ==========================
# COUNT FROM PREDICTED FRAMES
# ==========================

def count_rhonchi_from_predictions(predicted_labels):

    count = 0
    in_segment = False

    for val in predicted_labels:
        if val == 1 and not in_segment:
            count += 1
            in_segment = True
        elif val == 0:
            in_segment = False

    return count


# ==========================
# MAIN
# ==========================

def main():

    test_csv_folder = "output/test"
    audio_folder = r"E:\Research Project (Prof. Preeti Rao)\Top files by Rhonchi count"

    results = []

    for csv_file in sorted(os.listdir(test_csv_folder)):

        if not csv_file.endswith("_test.csv"):
            continue

        base = csv_file.replace("_test.csv", "")
        label_file = os.path.join(audio_folder, base + "_label_audacity.txt")

        if not os.path.exists(label_file):
            print(f"⚠ Missing label file for {base}")
            continue

        df = pd.read_csv(os.path.join(test_csv_folder, csv_file))

        true_count = count_rhonchi_from_label(label_file)

        pred_raw_count = count_rhonchi_from_predictions(
            df["predicted_label_raw"].values
        )

        pred_clean_count = count_rhonchi_from_predictions(
            df["predicted_label_0.15s"].values
        )

        results.append({
            "file": base,
            "true_count": true_count,
            "predicted_count_raw": pred_raw_count,
            "predicted_count_0.15s": pred_clean_count,
            "diff_raw": pred_raw_count - true_count,
            "diff_0.15s": pred_clean_count - true_count
        })

        print(f"{base}")
        print(f"  True count         : {true_count}")
        print(f"  Predicted          : {pred_clean_count}  (diff: {pred_clean_count - true_count:+d})")
        print()

    results_df = pd.DataFrame(results)
    results_df.to_csv("rhonchi_count_comparison.csv", index=False)

    print("=" * 45)
    print(f"Total files evaluated : {len(results)}")
    print(f"Total true count      : {results_df['true_count'].sum()}")
    print(f"Total predicted : {results_df['predicted_count_0.15s'].sum()}")
    print(f"Mean abs error : {results_df['diff_0.15s'].abs().mean():.2f}")
    print("\n Saved to rhonchi_count_comparison.csv")


if __name__ == "__main__":
    main()

steth_20180901_09_54_08
  True count         : 8
  Predicted          : 8  (diff: +0)

steth_20180921_15_12_58
  True count         : 8
  Predicted          : 12  (diff: +4)

steth_20181210_08_53_33
  True count         : 9
  Predicted          : 9  (diff: +0)

steth_20181210_08_53_52
  True count         : 8
  Predicted          : 9  (diff: +1)

steth_20181210_08_54_10
  True count         : 8
  Predicted          : 9  (diff: +1)

steth_20190128_12_21_53
  True count         : 10
  Predicted          : 7  (diff: -3)

steth_20190131_11_28_35
  True count         : 10
  Predicted          : 13  (diff: +3)

steth_20190618_15_19_21
  True count         : 13
  Predicted          : 5  (diff: -8)

steth_20190716_12_13_35
  True count         : 9
  Predicted          : 16  (diff: +7)

steth_20190716_12_15_24
  True count         : 8
  Predicted          : 12  (diff: +4)

trunc_2019-07-16-10-41-27-L1_10
  True count         : 8
  Predicted          : 9  (diff: +1)

trunc_2019-07-16-10-41-27-L5